# GraphRAG — Person B Smoke Test
Run each cell top to bottom.

> **Before starting:** Make sure the `GraphRAG/` folder is in `MyDrive/GraphRAG/`

In [ ]:
# ── Cell 1: Mount Google Drive ────────────────────────────────
# If this cell fails, go to Runtime > Disconnect and delete runtime, then reconnect.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, sys

PROJECT_ROOT = '/content/drive/MyDrive/GraphRAG'

# Verify the folder actually exists before proceeding
if not os.path.exists(PROJECT_ROOT):
    raise FileNotFoundError(
        f'Project folder not found at {PROJECT_ROOT}\n'
        'Upload the GraphRAG folder to MyDrive first.'
    )

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

print('Working directory:', os.getcwd())
print('Files found:', sorted(os.listdir('.')))
print('\n✅ Cell 1 passed — Drive mounted')

In [ ]:
# ── Cell 2: Install dependencies (fixed) ─────────────────────
# Installs to Colab's local runtime, NOT to Drive.
# Must re-run this cell every time you start a new Colab session.
# Split into small groups so one failure doesn't block everything.

import subprocess, sys

def install(packages, label):
    cmd = [sys.executable, '-m', 'pip', 'install', '--quiet'] + packages
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'❌ FAILED: {label}')
        print(result.stderr[-800:])  # show last 800 chars of error
    else:
        print(f'✅ {label}')

install(['pyyaml', 'python-dotenv'],                         'Core utilities')
install(['tiktoken'],                                        'Tokenizer')
install(['google-generativeai'],                             'Gemini API')
install(['groq'],                                            'Groq API')
install(['networkx'],                                        'NetworkX')
install(['chromadb'],                                        'ChromaDB')
install(['pymupdf'],                                         'PyMuPDF (PDF parser)')
install(['streamlit', 'pyvis'],                             'Streamlit + Pyvis')

# spaCy: install then download model separately
install(['spacy'],                                           'spaCy')
result = subprocess.run(
    [sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('✅ spaCy model downloaded')
else:
    print('❌ spaCy model download failed:', result.stderr[-300:])

# graspologic and leidenalg can conflict — try graspologic first, leidenalg as fallback
result_g = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', 'graspologic'],
    capture_output=True, text=True
)
if result_g.returncode == 0:
    print('✅ graspologic (primary Leiden backend)')
else:
    print('⚠️  graspologic failed, trying leidenalg fallback...')
    install(['leidenalg', 'igraph'], 'leidenalg + igraph (fallback Leiden backend)')

print('\n--- Installation complete ---')
print('If any ❌ above, re-run this cell once before panicking.')
print('If still failing, paste the error message for help.')

In [ ]:
# ── Cell 3: Verify imports before touching any project code ───
# Run this BEFORE Cell 4 to confirm all packages installed correctly.

failed = []

checks = [
    ('yaml',                    'pyyaml'),
    ('dotenv',                  'python-dotenv'),
    ('spacy',                   'spacy'),
    ('tiktoken',                'tiktoken'),
    ('google.generativeai',     'google-generativeai'),
    ('groq',                    'groq'),
    ('networkx',                'networkx'),
    ('chromadb',                'chromadb'),
    ('fitz',                    'pymupdf'),
]

for module, pkg in checks:
    try:
        __import__(module)
        print(f'  ✅ {pkg}')
    except ImportError as e:
        print(f'  ❌ {pkg}  →  {e}')
        failed.append(pkg)

# Check graspologic OR leidenalg
try:
    import graspologic
    print('  ✅ graspologic (Leiden backend)')
except ImportError:
    try:
        import leidenalg
        print('  ✅ leidenalg (Leiden backend fallback)')
    except ImportError:
        print('  ❌ Neither graspologic nor leidenalg installed — community detection will fail')
        failed.append('leiden-backend')

# Check spaCy model
try:
    import spacy
    spacy.load('en_core_web_sm')
    print('  ✅ spaCy en_core_web_sm model')
except OSError:
    print('  ❌ spaCy model not found — run: python -m spacy download en_core_web_sm')
    failed.append('spacy-model')

print()
if failed:
    print(f'❌ {len(failed)} package(s) missing: {failed}')
    print('Go back to Cell 2 and check which install step failed.')
else:
    print('✅ Cell 3 passed — all imports OK, safe to continue')

In [ ]:
# ── Cell 4: Set API keys ──────────────────────────────────────
GEMINI_API_KEY = 'paste_your_gemini_key_here'
GROQ_API_KEY   = 'paste_your_groq_key_here'

if GEMINI_API_KEY == 'paste_your_gemini_key_here':
    raise ValueError('Replace GEMINI_API_KEY with your actual key before running this cell.')

env_content = f'GEMINI_API_KEY={GEMINI_API_KEY}\nGROQ_API_KEY={GROQ_API_KEY}\n'
with open(f'{PROJECT_ROOT}/.env', 'w') as f:
    f.write(env_content)

print('.env written')
print('Gemini key set: True')
print('\n✅ Cell 4 passed — API keys saved')

In [ ]:
# ── Cell 5: Test config loader ────────────────────────────────
from src.config import cfg, GEMINI_API_KEY, CACHE_DIR, CHROMA_DIR, GRAPH_PATH

print('Config sections:', list(cfg.keys()))
print('Chunk size:', cfg['chunking']['chunk_size'])
print('LLM provider:', cfg['llm']['generation_provider'])
print('Cache dir:', CACHE_DIR)
print('Gemini key loaded:', bool(GEMINI_API_KEY))

assert 'chunking' in cfg
assert cfg['chunking']['chunk_size'] == 600
assert bool(GEMINI_API_KEY), 'Gemini key not loaded — check .env file'
print('\n✅ Cell 5 passed — config loads correctly')

In [ ]:
# ── Cell 6: Test chunker ──────────────────────────────────────
from src.chunker import chunk_document, split_into_sentences

sample_text = """
The transformer architecture was introduced in the paper Attention is All You Need.
It relies entirely on self-attention mechanisms, dispensing with recurrence and convolutions.
The encoder maps an input sequence to a sequence of continuous representations.
Given these representations, the decoder generates an output sequence one element at a time.
Self-attention allows each position in the sequence to attend to all positions in the previous layer.
Multi-head attention allows the model to jointly attend to information from different representation subspaces.
Position-wise feed-forward networks are applied to each position separately and identically.
The model uses residual connections around each sub-layer followed by layer normalization.
Positional encodings are added to the input embeddings to inject sequence order information.
""" * 15

sentences = split_into_sentences(sample_text)
print(f'Sentences found: {len(sentences)}')

chunks = chunk_document(sample_text, doc_id='test_001')
print(f'Chunks produced: {len(chunks)}')
for c in chunks[:3]:
    print(f"  [{c['chunk_index']}] tokens={c['token_count']}  sentences={c['sentences']}")

assert len(chunks) > 1
assert all(c['token_count'] <= 650 for c in chunks)
print('\n✅ Cell 6 passed — chunker works')

In [ ]:
# ── Cell 7: Test LLM client ───────────────────────────────────
from src.llm_client import LLMClient

llm = LLMClient(purpose='generation')
print('Provider:', llm.provider, '| Model:', llm.model)

response = llm.generate('In one sentence, what is a knowledge graph?')
print('Response:', response)

assert len(response) > 10, 'Response too short — check your API key'
print('\n✅ Cell 7 passed — LLM works')

In [ ]:
# ── Cell 8: Test vector store ─────────────────────────────────
from src.vector_store import VectorStore

vs = VectorStore()
print('Initial stats:', vs.stats())

vs.build_index(chunks)  # uses chunks from Cell 6
print('After indexing:', vs.stats())

results = vs.query('attention mechanism transformer')
print(f'\nQuery returned {len(results)} results')
for r in results[:2]:
    print(f"  score={r['score']}  text: {r['text'][:80]}...")

assert len(results) > 0
print('\n✅ Cell 8 passed — vector store works')

In [ ]:
# ── Cell 9: Test router ───────────────────────────────────────
from src.router import QueryRouter

router = QueryRouter(llm=llm)

test_queries = [
    ('What datasets were used in the BERT paper?',     'LOCAL'),
    ('What are the main themes across all papers?',    'GLOBAL'),
    ('Explain attention and its role in the field',    'HYBRID'),
]

print('Router results:')
for query, expected in test_queries:
    d = router.route(query)
    match = '✅' if d.route_type.value == expected else '⚠️ '
    print(f"  {match} [{d.route_type.value:6s}] conf={d.confidence:.2f}  '{query[:45]}'")

print('\n(⚠️  = router disagreed with expected label — not necessarily wrong)')
print('\n✅ Cell 9 passed — router runs')

In [ ]:
# ── Cell 10: Community detection on toy graph ─────────────────
import networkx as nx
from src.community_detection import CommunityDetector
from src.config import GRAPH_PATH
import os

# Build toy graph
G = nx.MultiDiGraph()
clusters = {
    'nlp':     ['transformer', 'attention', 'BERT', 'GPT', 'self_attention', 'encoder', 'decoder'],
    'cv':      ['CNN', 'ResNet', 'image_classification', 'convolution', 'pooling'],
    'training':['backpropagation', 'gradient_descent', 'Adam_optimizer', 'learning_rate'],
}

for nodes in clusters.values():
    for n in nodes:
        G.add_node(n, description=f'Concept: {n}', entity_type='concept', frequency=1)
    for i in range(len(nodes) - 1):
        G.add_edge(nodes[i], nodes[i+1],
                   relation_type='related_to', description='', weight=1.0, computed_weight=1.0)

# Sparse cross-cluster edges
G.add_edge('transformer', 'Adam_optimizer', relation_type='trained_with',
           description='', weight=0.5, computed_weight=0.5)
G.add_edge('BERT', 'backpropagation', relation_type='trained_with',
           description='', weight=0.5, computed_weight=0.5)

os.makedirs(GRAPH_PATH.parent, exist_ok=True)
nx.write_gml(G, str(GRAPH_PATH))
print(f'Toy graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

cd = CommunityDetector()
cd.load_graph()
cd.detect_all_levels()

print('\nCommunity stats:')
for res, stats in cd.stats().items():
    print(f'  Resolution {res}: {stats}')

members_1 = cd.get_community_members(1.0)
print(f'\nCommunities at resolution=1.0:')
for cid, members in members_1.items():
    print(f'  Community {cid}: {members}')

assert len(members_1) >= 2
print('\n✅ Cell 10 passed — community detection works')

In [ ]:
# ── Cell 11: Community summarizer ────────────────────────────
from src.community_summarizer import CommunitySummarizer

cs = CommunitySummarizer(detector=cd, llm=llm)

# Test on one community only to save API quota
first_comm_id = list(members_1.keys())[0]
first_members = members_1[first_comm_id]

print(f'Summarizing community {first_comm_id} ({len(first_members)} members)...')
summary = cs._summarize_community(first_comm_id, first_members, resolution=1.0)

print('Summary preview:')
print(summary['summary'][:300], '...')

assert len(summary['summary']) > 50
print('\n✅ Cell 11 passed — summarizer works')

In [ ]:
# ── Cell 12: Full end-to-end test ────────────────────────────
from src.graph_query import GraphQueryEngine
from src.query_engine import QueryEngine

print('Generating all community summaries (uses API quota)...')
cs.summarize_all(resolution=1.0)

graph_engine = GraphQueryEngine(summarizer=cs, llm=llm)
engine = QueryEngine(vector_store=vs, graph_engine=graph_engine, router=router, llm=llm)

test_cases = [
    ('What is self-attention?',                  'LOCAL'),
    ('What are the main themes in this corpus?', 'GLOBAL'),
    ('Explain transformers and their training',  'HYBRID'),
]

print('\nEnd-to-end results:')
print('=' * 55)
for query, forced_route in test_cases:
    result = engine.query(query, force_route=forced_route)
    print(f'\nRoute: {result["route"]} | Query: {query}')
    print(f'Answer: {result["answer"][:150]}...')
    assert result['answer']

print('\n✅ Cell 12 passed — full pipeline works')

In [ ]:
# ── Cell 13: Final summary ────────────────────────────────────
print('=' * 50)
print('ALL SMOKE TESTS PASSED')
print('=' * 50)
print('''
Verified:
  Cell 5  Config loader
  Cell 6  Sentence-aware chunker
  Cell 7  LLM client (Gemini)
  Cell 8  Vector store (ChromaDB)
  Cell 9  Hybrid query router
  Cell 10 Community detection (Leiden)
  Cell 11 Community summarizer
  Cell 12 Full end-to-end pipeline

Next steps:
  1. Wait for Person A to share knowledge_graph.gml
  2. Copy it to outputs/graphs/knowledge_graph.gml
  3. Re-run cells 10-12 on the real graph
  4. Run: streamlit run app.py
''')